# station_status_changes table

In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests

from pathlib import Path

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

# status changes:
- became empty
- recovered from empty
- became full
- went offline
- came online

I need to look back on the previous entry to see if there was a change in status. I use the LAG window function to do that 

In [ ]:
cursor.execute('''
SELECT *, LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_bikes
FROM availability_snapshots
limit 3;
''')
cursor.fetchall()

station_id not null  ref: > stations.station_id
change_type not null
detected_at not null
previous_value
new_value 
Snapshot_id


### To capture `became_empty`

In [ ]:
cursor.execute('''
    SELECT 
        station_id,
        'became_empty' AS change_type,
        captured_at AS detected_at,
        prev_num_bikes::text AS previous_value,
        num_bikes_available::text AS new_value,
        Snapshot_id
    FROM (SELECT *, 
            LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_bikes
          FROM availability_snapshots)tbl
    WHERE prev_num_bikes >0 AND num_bikes_available = 0
    limit 3;
        
        
''')

cursor.fetchall()

### capture became full


In [ ]:
cursor.execute('''
    SELECT 
        station_id,
        'became_full' AS change_type,
        captured_at AS detected_at,
        prev_num_docks::text AS previous_value,
        num_docks_available::text AS new_value,
        Snapshot_id
    FROM (SELECT *, 
            LAG(num_docks_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_docks
          FROM availability_snapshots)tbl
    WHERE prev_num_docks > 0 AND num_docks_available = 0
    limit 3;
        
        
''')

cursor.fetchall()

### recovered from empty

In [ ]:
cursor.execute('''
    SELECT 
        station_id,
        'recovered from empty' AS change_type,
        captured_at AS detected_at,
        prev_num_bikes::text AS previous_value,
        num_bikes_available::text AS new_value,
        Snapshot_id
    FROM (SELECT *, 
            LAG(num_bikes_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_bikes
          FROM availability_snapshots)tbl
    WHERE prev_num_bikes = 0 AND num_bikes_available > 0
    limit 3;
        
        
''')

cursor.fetchall()

### went offline
- detects when the station is partially operational to completely offline

In [ ]:
cursor.execute (
    '''
    SELECT 
        station_id,
        captured_at AS detected_at,
        'went offline' AS change_type,
        'online' AS previous_value,
        'offline' AS new_value,
        snapshot_id
    FROM (SELECT 
                *,
                LAG(is_renting) OVER (PARTITION BY station_id ORDER BY captured_at) AS is_renting_previous,
                LAG(is_returning) OVER (PARTITION BY station_id ORDER BY captured_at) AS is_returning_previous
            FROM availability_snapshots)tbl
    WHERE (is_renting_previous = True OR is_returning_previous = True) 
        AND is_renting = False 
        AND is_returning = False
    LIMIT 3;
    '''
)
cursor.fetchall()

In [ ]:
cursor.execute('''
    SELECT * FROM availability_snapshots LIMIT 3;
    ''')
cursor.fetchall()

### came online

In [ ]:
cursor.execute (
    '''
    SELECT 
        station_id,
        captured_at AS detected_at,
        'came online' AS change_type,
        'offline' AS previous_value,
        'online' AS new_value,
        snapshot_id
    FROM (SELECT 
                *,
                LAG(is_renting) OVER (PARTITION BY station_id ORDER BY captured_at) AS is_renting_previous,
                LAG(is_returning) OVER (PARTITION BY station_id ORDER BY captured_at) AS is_returning_previous
            FROM availability_snapshots)tbl
    WHERE (is_renting_previous = False AND is_returning_previous = FAlse) 
        AND (is_renting = True OR is_returning = True)
    LIMIT 3;
    '''
)
cursor.fetchall()

In [ ]:
cursor.execute('''

SELECT station_id, 
        'became_full' AS change_type,  
        captured_at AS detected_at,
        prev_num_docks::text AS previous_value,
        num_docks_available::text AS new_value,
        Snapshot_id
    		FROM (SELECT *, 
           LAG(num_docks_available) OVER (PARTITION BY 
station_id ORDER BY captured_at) AS prev_num_docks
          			FROM availability_snapshots)tbl
    		WHERE prev_num_docks > 0 AND num_docks_available = 0
UNION ALL

	SELECT 
        station_id,
        'became_full' AS change_type,
        captured_at AS detected_at,
        prev_num_docks::text AS previous_value,
        num_docks_available::text AS new_value,
        Snapshot_id
    		FROM (SELECT *, 
           LAG(num_docks_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_docks
          			FROM availability_snapshots)tbl
    		WHERE prev_num_docks > 0 AND num_docks_available = 0
UNION ALL

	SELECT 
        station_id,
        'recovered from empty' AS change_type,
        captured_at AS detected_at,
        prev_num_bikes::text AS previous_value,
        num_bikes_available::text AS new_value,
        Snapshot_id
    		FROM (SELECT *, 
            		LAG(num_bikes_available) OVER (PARTITION BY station_id 
ORDER BY captured_at) AS prev_num_bikes
          			FROM availability_snapshots)tbl
   		WHERE prev_num_bikes = 0 AND num_bikes_available > 0
UNION ALL

    SELECT 
        station_id,
        'went offline' AS change_type,
        captured_at AS detected_at,      
        'online' AS previous_value,
        'offline' AS new_value,
        snapshot_id
    FROM (SELECT 
                *,
                LAG(is_renting) OVER (PARTITION BY station_id ORDER BY 
                    captured_at) AS is_renting_previous,
                LAG(is_returning) OVER (PARTITION BY station_id ORDER BY 
                    captured_at) AS is_returning_previous
            FROM availability_snapshots)tbl
    WHERE (is_renting_previous = True OR is_returning_previous = True) 
        AND is_renting = False 
        AND is_returning = False
UNION ALL

    SELECT 
        station_id,
        'came online' AS change_type,
        captured_at AS detected_at,
        'offline' AS previous_value,
        'online' AS new_value,
        Snapshot_id
    FROM (SELECT 
                *,
                LAG(is_renting) OVER (PARTITION BY station_id 
                    ORDER BY captured_at) AS is_renting_previous,
                LAG(is_returning) OVER (PARTITION BY station_id ORDER 
                    BY captured_at) AS is_returning_previous
            FROM availability_snapshots)tbl
    WHERE (is_renting_previous = False AND is_returning_previous = False) AND (is_renting = True OR is_returning = True)
    LIMIT 3;
''')
cursor.fetchall()

### Insert into 



cursor.execute('''
INSERT INTO station_status_changes (station_id, change_type, detected_at, previous_value, new_value, snapshot_id
    
    	SELECT 
            station_id,
            'became_full' AS change_type,
            captured_at AS detected_at,
            prev_num_docks::text AS previous_value,
            num_docks_available::text AS new_value,
            Snapshot_id
        		FROM (SELECT *, 
               LAG(num_docks_available) OVER (PARTITION BY station_id ORDER BY captured_at) AS prev_num_docks
              			FROM availability_snapshots)tbl
        		WHERE prev_num_docks > 0 AND num_docks_available = 0
    UNION ALL
    
    	SELECT 
            station_id,
            'recovered from empty' AS change_type,
            captured_at AS detected_at,
            prev_num_bikes::text AS previous_value,
            num_bikes_available::text AS new_value,
            Snapshot_id
        		FROM (SELECT *, 
                		LAG(num_bikes_available) OVER (PARTITION BY station_id 
    ORDER BY captured_at) AS prev_num_bikes
              			FROM availability_snapshots)tbl
       		WHERE prev_num_bikes = 0 AND num_bikes_available > 0
    UNION ALL
    
        SELECT 
            station_id,
            'went offline' AS change_type,
            captured_at AS detected_at,      
            'online' AS previous_value,
            'offline' AS new_value,
            snapshot_id
        FROM (SELECT 
                    *,
                    LAG(is_renting) OVER (PARTITION BY station_id ORDER BY 
                        captured_at) AS is_renting_previous,
                    LAG(is_returning) OVER (PARTITION BY station_id ORDER BY 
                        captured_at) AS is_returning_previous
                FROM availability_snapshots)tbl
        WHERE (is_renting_previous = True OR is_returning_previous = True) 
            AND is_renting = False 
            AND is_returning = False
    UNION ALL
    
        SELECT 
            station_id,
            'came online' AS change_type,
            captured_at AS detected_at,
            'offline' AS previous_value,
            'online' AS new_value,
            Snapshot_id
        FROM (SELECT 
                    *,
                    LAG(is_renting) OVER (PARTITION BY station_id 
                        ORDER BY captured_at) AS is_renting_previous,
                    LAG(is_returning) OVER (PARTITION BY station_id ORDER 
                        BY captured_at) AS is_returning_previous
                FROM availability_snapshots)tbl
        WHERE (is_renting_previous = False AND is_returning_previous = False) AND (is_renting = True OR is_returning = True);
''')

In [ ]:
cursor.execute(
    '''
    select COUNT(*) from station_status_changes
    where previous_value is null;
    '''
)
cursor.fetchone()

In [ ]:
cursor.execute("""
SELECT change_id FROM (
        SELECT change_id, 
               LEAD(detected_at) OVER (PARTITION BY station_id ORDER BY detected_at) - detected_at AS gap
        FROM station_status_changes
        WHERE change_type IN ('became empty', 'recovered from empty')
    ) sub
    WHERE gap < INTERVAL '2 minutes' LIMIT 10;
""")
cursor.fetchall()

In [ ]:
conn.close()
print('Connection is closed')